In [1]:
"""
COMPONENT 2: MATHEMATICAL MODELING
- Physical Equations
- Statistical Models
- Uncertainty Quantification
"""

class MathematicalModel:
    """
    Core mathematical model for PV production under climate change
    """
    
    def __init__(self):
        # Physical constants
        self.P_STC = 1000  # kW (rated power at Standard Test Conditions)
        self.G_STC = 1000  # W/m² (irradiance at STC)
        self.T_STC = 25    # °C (temperature at STC)
        self.beta = 0.004  # Temperature coefficient (per °C)
        self.eta_inv = 0.96  # Inverter efficiency
        self.eta_sys = 0.85   # System efficiency (wiring, losses, etc.)
        
    def temperature_model(self, T_amb, T_NOCT=45, G=800):
        """
        Equation 1: Cell temperature model
        T_cell = T_amb + (T_NOCT - 20) * (G / 800)
        
        Where:
        - T_cell: Cell temperature (°C)
        - T_amb: Ambient temperature (°C)
        - T_NOCT: Nominal Operating Cell Temperature (°C)
        - G: Irradiance (W/m²)
        """
        T_cell = T_amb + (T_NOCT - 20) * (G / 800)
        return T_cell
    
    def temperature_loss_model(self, T_cell):
        """
        Equation 2: Temperature loss factor
        f_temp = 1 - β * (T_cell - T_STC)
        
        Where β is the temperature coefficient
        """
        f_temp = 1 - self.beta * (T_cell - self.T_STC)
        return np.maximum(f_temp, 0.7)  # Cap at 70% efficiency
    
    def irradiance_model(self, G, cloud_cover):
        """
        Equation 3: Effective irradiance model
        G_eff = G * (1 - α * CC) * (1 + γ * sin(θ))
        
        Where:
        - G_eff: Effective irradiance (W/m²)
        - CC: Cloud cover (0-1)
        - α: Cloud attenuation coefficient (typically 0.3-0.7)
        - θ: Solar zenith angle
        - γ: Angular loss coefficient
        """
        alpha = 0.5  # Cloud attenuation coefficient
        gamma = 0.05  # Angular loss coefficient
        
        # Simplified model (assuming optimal angle)
        G_eff = G * (1 - alpha * cloud_cover)
        return np.maximum(G_eff, 0)
    
    def extreme_events_model(self, events_base, severity, T_amb, G):
        """
        Equation 4: Extreme events impact model
        f_extreme = exp(-∑(λ_i * s_i))
        
        Where:
        - λ_i: Event type impact factor
        - s_i: Event severity
        """
        # Event type impact factors
        impact_factors = {
            'heat_wave': 0.15,    # 15% reduction during heat wave
            'storm': 0.25,         # 25% reduction during storm
            'haze': 0.10,          # 10% reduction during haze
            'wildfire': 0.30       # 30% reduction during wildfire
        }
        
        # Simplified: average impact based on severity
        avg_impact = 0.2 * (severity / 3)  # Normalized severity
        
        # Poisson process for event occurrence
        f_extreme = np.exp(-events_base * avg_impact)
        return f_extreme
    
    def pv_power_model(self, G_eff, T_cell, events_factor):
        """
        Equation 5: Complete PV power output model
        P = P_STC * (G_eff/G_STC) * [1 - β(T_cell - T_STC)] * η_inv * η_sys * f_extreme
        
        This is the core equation combining all factors
        """
        # Normalized irradiance factor
        f_irrad = G_eff / self.G_STC
        
        # Temperature factor
        f_temp = 1 - self.beta * (T_cell - self.T_STC)
        
        # Combined power output
        P = self.P_STC * f_irrad * f_temp * self.eta_inv * self.eta_sys * events_factor
        
        return np.maximum(P, 0)
    
    def stochastic_model(self, mu, sigma, n_samples=1000, distribution='normal'):
        """
        Equation 6: Stochastic model for uncertainty
        P_stochastic = P_deterministic + ε
        
        Where ε follows specified distribution
        """
        if distribution == 'normal':
            epsilon = np.random.normal(0, sigma, n_samples)
        elif distribution == 'lognormal':
            epsilon = np.random.lognormal(0, sigma, n_samples) - np.exp(sigma**2/2)
        elif distribution == 'weibull':
            epsilon = np.random.weibull(2, n_samples) * sigma - sigma * np.math.gamma(1 + 1/2)
        
        return mu + epsilon
    
    def time_series_model(self, P_base, years, seasonal_amplitude=0.1, noise_level=0.05):
        """
        Equation 7: Time series model with seasonality
        P(t) = P_base * (1 + A * sin(2πt/T + φ)) * (1 + ε)
        
        Where:
        - A: Seasonal amplitude
        - T: Period (12 months)
        - φ: Phase shift
        - ε: Random noise
        """
        t = np.arange(len(years) * 12) / 12  # Monthly data
        seasonal = 1 + seasonal_amplitude * np.sin(2 * np.pi * t)
        noise = 1 + np.random.normal(0, noise_level, len(t))
        
        # Repeat base annual values for monthly data
        P_annual = np.repeat(P_base, 12)
        P_monthly = P_annual[:len(t)] * seasonal * noise
        
        return P_monthly, t

# Model validation and simulation
def simulate_model_scenarios():
    """
    Run model simulations for different scenarios
    """
    model = MathematicalModel()
    
    # Scenario definitions
    scenarios = {
        'Baseline': {'temp_trend': 0.02, 'cloud_trend': 0.0005, 'event_trend': 0.01},
        'Moderate': {'temp_trend': 0.03, 'cloud_trend': 0.001, 'event_trend': 0.02},
        'Severe': {'temp_trend': 0.05, 'cloud_trend': 0.002, 'event_trend': 0.03}
    }
    
    years = np.arange(2024, 2075)
    results = {}
    
    for scenario_name, params in scenarios.items():
        # Generate climate variables
        T_amb = 25 + params['temp_trend'] * (years - 2024)
        cloud_cover = 0.3 + params['cloud_trend'] * (years - 2024)
        G = 800 * (1 - 0.001 * (years - 2024))  # Declining irradiance
        events = 2 + params['event_trend'] * (years - 2024)
        
        # Calculate PV output
        P_output = []
        for i in range(len(years)):
            T_cell = model.temperature_model(T_amb[i], G=G[i])
            G_eff = model.irradiance_model(G[i], cloud_cover[i])
            events_factor = model.extreme_events_model(events[i], 3, T_amb[i], G[i])
            P = model.pv_power_model(G_eff, T_cell, events_factor)
            P_output.append(P)
        
        results[scenario_name] = np.array(P_output)
    
    return results, years

# Run simulations
scenario_results, years = simulate_model_scenarios()

# Plot scenario comparison
plt.figure(figsize=(12, 6))
for scenario, P in scenario_results.items():
    plt.plot(years, P, label=scenario, linewidth=2)
plt.xlabel('Year')
plt.ylabel('PV Power Output (kW)')
plt.title('Scenario Analysis: Climate Impact on PV Production')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('scenario_analysis.png', dpi=300)
plt.show()

NameError: name 'np' is not defined